<a href="https://colab.research.google.com/github/Lee-Minsoo-97/Sales-Data-Prediction/blob/gemini_original/iHerb_Sales_Pred_ML_Project_v2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## v2.0: XGBoost 하이퍼파라미터 튜닝

In [ ]:
# XGBoost와 Optuna 라이브러리 설치
!pip install xgboost optuna

import xgboost as xgb
import optuna
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import pandas as pd

# --- 1. 데이터 불러오기 ---
# 이전과 동일하게 모델링용 데이터를 불러옵니다.
file_path = '/content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv'
df_model = pd.read_csv(file_path, parse_dates=['Date'])

print("✅ 모델링용 데이터를 성공적으로 불러왔습니다.")

# --- 2. XGBoost 튜닝을 위한 함수 정의 ---
def objective_xgb_cv(trial):
    # XGBoost를 위한 하이퍼파라미터 탐색 범위 지정
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'tree_method': 'gpu_hist',  # GPU 사용을 위한 파라미터
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }

    tscv = TimeSeriesSplit(n_splits=5)
    maes = []

    X = df_model.drop(columns=['Date', 'SKU', 'UPC Code', 'Product Description', 'Sales'])
    y = df_model['Sales']

    for train_index, val_index in tscv.split(X):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, verbose=False)
        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        maes.append(mae)

    return np.mean(maes)

# --- 3. Optuna 튜닝 실행 ---
print("\n🚀 XGBoost 하이퍼파라미터 튜닝을 시작합니다... (GPU 사용)")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb_cv, n_trials=100)

print("\n✅ XGBoost 교차 검증 튜닝이 완료되었습니다.")
print(f"XGBoost 평균 MAE (교차 검증): {study_xgb.best_value:.2f}")
print("\n--- XGBoost 최적 하이퍼파라미터 조합 ---")
print(study_xgb.best_trial.params)

[I 2025-10-10 20:13:36,576] A new study created in memory with name: no-name-1e5c3abd-9bbb-4c8a-ae7a-af3349703864


✅ 모델링용 데이터를 성공적으로 불러왔습니다.

🚀 XGBoost 하이퍼파라미터 튜닝을 시작합니다... (GPU 사용)


Streaming output truncated to the last 5000 lines.
  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
[I 2025-10-10 20:14:40,030] Trial 1 finished with value: 29.72412109375 and parameters: {'n_estimators': 2324, 'learning_rate': 0.08081803132379631, 'max_depth': 7, 'subsample': 0.97533962027932, 'colsample_bytree': 0.8513176952658023}. Best is trial 1 with value: 29.72412109375.
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:14:40] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [20:14:41] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead


✅ XGBoost 교차 검증 튜닝이 완료되었습니다.
XGBoost 평균 MAE (교차 검증): 27.03

--- XGBoost 최적 하이퍼파라미터 조합 ---
{'n_estimators': 795, 'learning_rate': 0.015499124099838158, 'max_depth': 4, 'subsample': 0.9118508428731791, 'colsample_bytree': 0.8944788408260789}


## Overfitting 방지 ver.

In [ ]:
import xgboost as xgb
import joblib
from sklearn.metrics import mean_absolute_error
import pandas as pd

# 1. 조기 종료를 위한 검증(Validation) 데이터 분리 (이전과 동일)
test_start_date = '2025-07-01'
train_df = df_model[df_model['Date'] < test_start_date]
test_df = df_model[df_model['Date'] >= test_start_date]

val_start_date = '2025-06-01'
train_sub_df = train_df[train_df['Date'] < val_start_date]
val_df = train_df[train_df['Date'] >= val_start_date]

cols_to_drop = ['Date', 'SKU', 'UPC Code', 'Product Description', 'Sales']

X_train_sub = train_sub_df.drop(columns=cols_to_drop)
y_train_sub = train_sub_df['Sales']
X_val = val_df.drop(columns=cols_to_drop)
y_val = val_df['Sales']

# 2. 최적 파라미터로 XGBoost 모델 훈련
best_params_xgb = study_xgb.best_trial.params
best_params_xgb['tree_method'] = 'hist'
best_params_xgb['device'] = 'cuda'
best_params_xgb['objective'] = 'reg:absoluteerror'
best_params_xgb['eval_metric'] = 'mae'
best_params_xgb['n_estimators'] = 5000

# ★★★ 수정된 부분 ★★★
# final_model_v2_0_xgb = xgb.XGBRegressor(**best_params_xgb, random_state=42) # 기존 코드
final_model_v2_0_xgb = xgb.XGBRegressor(**best_params_xgb,
                                        early_stopping_rounds=50, # 모델 생성 시 조기 종료 옵션 추가
                                        random_state=42)

print("\n🚀 최종 XGBoost 모델 훈련을 시작합니다... (조기 종료 적용)")

# .fit() 함수에서는 early_stopping_rounds 제거
final_model_v2_0_xgb.fit(X_train_sub, y_train_sub,
                         eval_set=[(X_val, y_val)],
                         verbose=100)
# --------------------

print("✅ 훈련이 완료되었습니다.")

# 3. 최종 성능 평가 및 모델 저장
X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df['Sales']
preds_xgb = final_model_v2_0_xgb.predict(X_test)
mae_xgb = mean_absolute_error(y_test, preds_xgb)

print("\n--- 최종 XGBoost 모델 예측 성능 ---")
print(f"테스트 데이터 최종 MAE: {mae_xgb:.2f}")


🚀 최종 XGBoost 모델 훈련을 시작합니다... (조기 종료 적용)
[0]	validation_0-mae:43.12715
[100]	validation_0-mae:26.92721
[200]	validation_0-mae:23.85591
[300]	validation_0-mae:22.82131
[305]	validation_0-mae:22.70881
✅ 훈련이 완료되었습니다.

--- 최종 XGBoost 모델 예측 성능 ---
테스트 데이터 최종 MAE: 35.79


In [ ]:
# 모델 저장
xgb_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/xgb_model_v2_0.pkl'
joblib.dump(final_model_v2_0_xgb, xgb_model_filename)
print(f"\n✅ XGBoost 모델이 경로에 저장되었습니다: {xgb_model_filename}")


✅ XGBoost 모델이 경로에 저장되었습니다: /content/drive/MyDrive/Colab Notebooks/Trained Models/xgb_model_v2_0.pkl
